No,２

In [1]:
import os
import shutil
import csv
import re
from pathlib import Path
import tkinter as tk
from tkinter import filedialog, messagebox
import stat
import time

選択したメインフォルダ内のサブフォルダ内のサブサブフォルダ内にあるファイルを拡張子ごとにフォルダ分け

In [2]:
# === 【1工程目】ダイアログでのフォルダ指定と、拡張子ごとの仕分け ===
def get_folder_path():
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    folder_path = filedialog.askdirectory(title="整理したい対象のメインフォルダを選択してください")
    return folder_path

print("ダイアログボックスを開いています...")
main_folder_path = get_folder_path()

if main_folder_path and os.path.isdir(main_folder_path):
    print(f"指定されたメインフォルダ: {main_folder_path}\nファイルの移動を開始します...")
    try:
        # 1. 移動対象のすべてのファイルのパスをリストアップ
        files_to_move = []
        for current_dir, dirs, files in os.walk(main_folder_path):
            for filename in files:
                # 隠しファイル（.で始まるファイル）は除外
                if not filename.startswith('.'):
                    files_to_move.append(os.path.join(current_dir, filename))

        # 2. ファイルを移動
        for original_file_path in files_to_move:
            file_dir = os.path.dirname(original_file_path)
            filename = os.path.basename(original_file_path)
            file_extension = os.path.splitext(filename)[1]
            
            if file_extension:
                extension_folder_name = file_extension[1:]
                relative_path = os.path.relpath(file_dir, main_folder_path)
                main_dest_folder = os.path.join(main_folder_path, extension_folder_name)
                
                
                if relative_path == ".":
                    final_dest_dir = main_dest_folder
                else:
                    final_dest_dir = os.path.join(main_dest_folder, relative_path)
                
                if not os.path.exists(final_dest_dir):
                    os.makedirs(final_dest_dir)
                
                final_dest_path = os.path.join(final_dest_dir, filename)
                shutil.move(original_file_path, final_dest_path)

        print("1工程目：ファイルの移動が完了しました。次のセル（2工程目）を実行してください。")

    except Exception as e:
        print(f"エラーが発生しました: {e}")

elif not main_folder_path:
    print("処理がキャンセルされたか、フォルダが選択されませんでした。")
else:
    print(f"エラー: 指定されたパス「{main_folder_path}」が見つかりません。")

ダイアログボックスを開いています...
指定されたメインフォルダ: C:/Users/0uh2j/OneDrive/Desktop/実験データ2026/本実験データ/202603-4 - 本実験圧力データ/202606-本実験データ再々
ファイルの移動を開始します...
1工程目：ファイルの移動が完了しました。次のセル（2工程目）を実行してください。


不要になった空のフォルダの削除

In [3]:
# === 【2工程目】不要になった元の空フォルダ（サブフォルダごと一括）の削除 ===


# 読み取り専用ファイルが原因で削除できない場合に、権限を強制変更して削除を再試行する関数
def force_remove_readonly(func, path, exc_info):
    try:
        os.chmod(path, stat.S_IWRITE) # 書き込み権限を付与
        func(path)                    # 再度削除を実行
    except Exception as e:
        pass # それでもダメな場合はスキップ

# 1工程目で 'main_folder_path' が正しく取得できているか確認
if 'main_folder_path' in locals() and main_folder_path and os.path.isdir(main_folder_path):
    print("不要になった元のサブフォルダの削除を開始します...")
    try:
        for item_name in os.listdir(main_folder_path):
            item_path = os.path.join(main_folder_path, item_name)
            
            if os.path.isdir(item_path) and not item_name.startswith('.'):
                
                # フォルダ内に「通常のファイル」が残っていないかチェック
                has_visible_file = False
                for root, dirs, files in os.walk(item_path):
                    for f in files:
                        if not f.startswith('.'):
                            has_visible_file = True
                            break
                    if has_visible_file:
                        break
                
                # 通常のファイルが一切残っていなければ、フォルダごと一括削除
                if not has_visible_file:
                    try:
                        # 1工程目の直後だとOSがロックしていることがあるため、0.5秒待機する
                        time.sleep(0.5)
                        
                        # onerror=force_remove_readonly を付けて、読み取り専用でも強制削除
                        shutil.rmtree(item_path, onerror=force_remove_readonly)
                        
                        # 削除できたか最終確認
                        if not os.path.exists(item_path):
                            print(f"削除したフォルダ: {item_path}")
                        else:
                            print(f"※完全には削除しきれませんでした: {item_path}")
                            
                    except OSError as e:
                        # 今回はOSの生のエラーメッセージも出力して原因を特定しやすくします
                        print(f"※削除スキップ: {item_path}\n  -> エラー詳細: {e}")

        print("\nすべての処理が完了しました！")

    except Exception as e:
        print(f"エラーが発生しました: {e}")
else:
    print("エラー: メインフォルダが認識されていません。先に1工程目のセルを実行してください。")

不要になった元のサブフォルダの削除を開始します...
削除したフォルダ: C:/Users/0uh2j/OneDrive/Desktop/実験データ2026/本実験データ/202603-4 - 本実験圧力データ/202606-本実験データ再々\0.5mm

すべての処理が完了しました！


ファイル分け最終段階：各拡張子フォルダがメインフォルダの直下に来るように整理

In [4]:
#拡張子ごとに成形条件フォルダの整理


def reorganize_deep_folders():
    # 1. フォルダ選択ダイアログを表示
    root_window = tk.Tk()
    root_window.withdraw()
    
    selected_path = filedialog.askdirectory(title="メインフォルダを選択してください")
    if not selected_path:
        print("キャンセルされました。")
        return

    root = Path(selected_path)
    count = 0

    # 2. 処理の実行
    # 探索の深さ: メイン(root) / サブ(*) / サブサブ(*) / データフォルダ(*)
    # iterdir() や glob("*/*/*") を組み合わせて、データフォルダのパスを取得
    for sub_dir in root.iterdir():
        if not sub_dir.is_dir(): continue
        
        for sub_sub_dir in sub_dir.iterdir():
            if not sub_sub_dir.is_dir(): continue
            
            # ここで「サブサブフォルダ名」を拡張子名として取得
            ext_name = sub_sub_dir.name
            
            for data_folder in sub_sub_dir.iterdir():
                if data_folder.is_dir():
                    # --- 移動先の決定 ---
                    # 新構造: メイン / 拡張子名(ext_name) / 元のサブ名(sub_dir.name) / データフォルダ
                    new_parent_dir = root / ext_name / sub_dir.name
                    
                    # 移動先フォルダを作成（存在しない場合のみ）
                    new_parent_dir.mkdir(parents=True, exist_ok=True)
                    
                    # 移動実行
                    try:
                        shutil.move(str(data_folder), str(new_parent_dir / data_folder.name))
                        count += 1
                    except Exception as e:
                        print(f"Error moving {data_folder.name}: {e}")

    # 3. 完了通知
    messagebox.showinfo("完了", f"処理が終了しました。\n移動したデータフォルダ数: {count}")

if __name__ == "__main__":
    reorganize_deep_folders()

キャンセルされました。
